In [1]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
wavs = sorted((root / "data" / "raw_audios").glob("*.wav"))
for i, p in enumerate(wavs):
    print(i, p.name)

0 OyQOvi9Qhxs.wav
1 abuser.wav


In [2]:
audio_path = wavs[0]  # same file as in step 4

MIN_SEC, TARGET_SEC, MAX_SEC = 2.0, 8.0, 15.0  # clip length limits
HARD_GAP_SEC = 1.0      # a silence this long always ends a clip
PAD_SEC = 0.1           # silence kept before the first and after the last word
MIN_TURN_SEC = 0.2      # shorter diarization turns are noise, ignore them
WHOLE_SENTENCES = True  # drop half sentences at the edges (needs punctuation)

diar_dir = root / "outputs" / "diarization" / audio_path.stem / "nemotron"
model_audio = diar_dir / "audio_16k_mono.wav"
out_dir = root / "outputs" / "chunks" / audio_path.stem
out_dir.mkdir(parents=True, exist_ok=True)

# RTTM line: SPEAKER <file> 1 <start> <duration> <NA> <NA> <speaker> <NA> <NA>
turns = []
for line in (diar_dir / f"{audio_path.stem}.rttm").read_text().splitlines():
    p = line.split()
    s, d, spk = float(p[3]), float(p[4]), p[7]
    if d >= MIN_TURN_SEC:
        turns.append((s, s + d, spk))
turns.sort()
total_sec = sf.info(model_audio).duration
print(len(turns), "turns,", round(total_sec, 1), "s")

19 turns, 69.2 s


In [6]:
turns

[(0.06, 3.0, 'speaker_0'),
 (3.05, 3.7699999999999996, 'speaker_0'),
 (4.09, 5.08, 'speaker_1'),
 (5.17, 8.98, 'speaker_0'),
 (9.23, 11.36, 'speaker_0'),
 (11.61, 16.53, 'speaker_0'),
 (17.2, 26.759999999999998, 'speaker_0'),
 (27.65, 34.08, 'speaker_0'),
 (30.7, 31.78, 'speaker_2'),
 (33.6, 35.26, 'speaker_2'),
 (34.64, 43.44, 'speaker_0'),
 (43.8, 44.41, 'speaker_2'),
 (44.94, 48.08, 'speaker_0'),
 (48.19, 49.4, 'speaker_0'),
 (49.06, 51.160000000000004, 'speaker_2'),
 (50.93, 58.09, 'speaker_0'),
 (53.78, 54.4, 'speaker_2'),
 (57.81, 58.81, 'speaker_2'),
 (58.64, 65.98, 'speaker_0')]

In [4]:
WINDOW_SEC = 60  # longest piece sent to the ASR at once


def make_windows(turns, total_sec, window_sec):
    """Cut the audio into pieces of at most window_sec, in silences where nobody speaks."""
    silences, busy = [], 0.0
    for s, e, _ in turns:
        if s - busy >= 0.3:
            silences.append((busy + s) / 2)
        busy = max(busy, e)
    windows, start = [], 0.0
    while total_sec - start > window_sec:
        fits = [m for m in silences if start + 1 < m <= start + window_sec]
        cut = fits[-1] if fits else start + window_sec
        windows.append((start, cut))
        start = cut
    windows.append((start, total_sec))
    return windows


windows = make_windows(turns, total_sec, WINDOW_SEC)
print(len(windows), "windows:", [(round(a, 1), round(b, 1)) for a, b in windows])

2 windows: [(0.0, 44.7), (44.7, 69.2)]


In [5]:
windows

[(0.0, 44.675), (44.675, 69.1955)]

In [7]:
import nemo.collections.asr as nemo_asr

asr = nemo_asr.models.ASRModel.from_pretrained("nvidia/stt_uz_fastconformer_hybrid_large_pc")
asr.eval()

win_dir = out_dir / "asr_windows"
win_dir.mkdir(exist_ok=True)
audio16, sr16 = sf.read(model_audio, dtype="float32")
paths = []
for i, (a, b) in enumerate(windows):
    path = win_dir / f"{i:04d}.wav"
    sf.write(path, audio16[int(a * sr16):int(b * sr16)], sr16)
    paths.append(str(path))

hyps = asr.transcribe(paths, batch_size=1, timestamps=True)
if isinstance(hyps, tuple):  # older NeMo returns (best, all)
    hyps = hyps[0]

# word times are counted from the window start -> add it
words = [
    {"word": w["word"], "start": round(a + w["start"], 3), "end": round(a + w["end"], 3)}
    for (a, _), h in zip(windows, hyps)
    for w in h.timestamp["word"]
]
(out_dir / "words.json").write_text(json.dumps(words, ensure_ascii=False, indent=1), encoding="utf-8")
print(len(words), "words")
for w in words[:15]:
    print(f'{w["start"]:7.2f} - {w["end"]:7.2f}  {w["word"]}')

/Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[NeMo I 2026-11-18 19:42:48 mixins:197] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-11-18 19:42:49 modelPT:192] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /workspace-mnt/final-train-manifests/MCV/train_manifest_final.json
    - /workspace-mnt/final-train-manifests/FLEURS/train_manifest_final.json
    - /workspace-mnt/final-train-manifests/UZBEKVOICE/train_manifest_final.json
    sample_rate: 16000
    batch_size: 32
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-11-18 19:42:49 modelPT:199] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the

[NeMo I 2026-11-18 19:42:49 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-11-18 19:42:49 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-18 19:42:49 label_looping_base:144] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-11-18 19:42:49 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-18 19:42:49 label_looping_base:144] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-11-18 19:42:49 save_restore_connector:309] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /Users/macbook/.cache/huggingface/hub/models--nvidia--stt_uz_fastconformer_hybrid_large_pc/snapshots/cd32f63d620a7b0cb461ed00ecfc45c852eb0bef/stt_uz_fastconformer_hybrid_large_pc.nemo.
[NeMo I 2026-11-18 19:42:49 hybrid_rnnt_ctc_models:148] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[idx].timestep['word'/'segment'/'char']
[NeMo I 2026-11-18 19:42:49 rnnt_models:233] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-18 19:42:49 label_looping_base:144] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available
[NeMo W 2026-11-18 19:42:49 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-11-18 19:42:49 dataloader:883] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:00, ?it/s][NeMo W 2026-11-18 19:42:49 nemo_logging:365] /Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set

73 words
   0.16 -    0.56  Oʻzbekistonda
   0.72 -    1.12  navin
   1.20 -    1.44  yutgan
   1.60 -    1.68  odam
   1.84 -    2.16  borligini
   2.24 -    2.64  bilarmidi
   6.48 -    6.80  hoziram
   6.96 -    7.36  keyingi
   7.52 -    7.76  oʻyin
   8.80 -   10.00  borgan
  10.08 -   10.64  professionallik
  12.00 -   12.16  bitta
  12.32 -   12.80  turnirda
  14.08 -   14.40  ismlari
  15.60 -   17.68  millatchi


In [8]:
def overlap(a1, b1, a2, b2):
    """Seconds shared by two time ranges."""
    return max(0.0, min(b1, b2) - max(a1, a2))


def word_speaker(w, turns, near=0.3):
    """Speaker of a word, "OVERLAP" if two people talk during it, None if nobody is near."""
    talk = Counter()
    for s, e, spk in turns:
        talk[spk] += overlap(w["start"], w["end"], s, e)
    talk = +talk  # keep only speakers with time > 0
    if len(talk) > 1 and sorted(talk.values())[-2] > 0.05:
        return "OVERLAP"
    if talk:
        return talk.most_common(1)[0][0]
    dist, spk = min((max(s - w["end"], w["start"] - e), spk) for s, e, spk in turns)
    return spk if dist <= near else None


words = json.loads((out_dir / "words.json").read_text(encoding="utf-8"))
for i, w in enumerate(words):
    w["i"] = i
    w["speaker"] = word_speaker(w, turns)
print(Counter(w["speaker"] for w in words))

Counter({'speaker_0': 66, 'OVERLAP': 5, 'speaker_2': 2})


In [9]:
SENT_END = (".", "?", "!", "…")
CLAUSE_END = (",", ";", ":", "-", "—")


def split_runs(words):
    """Groups of neighbouring words from ONE speaker, with no long silence or overlap inside."""
    runs, cur = [], []
    for w in words:
        bad = w["speaker"] in (None, "OVERLAP")
        if cur and (bad or w["speaker"] != cur[-1]["speaker"] or w["start"] - cur[-1]["end"] > HARD_GAP_SEC):
            runs.append(cur)
            cur = []
        if not bad:
            cur.append(w)
    if cur:
        runs.append(cur)
    return runs


def whole_sentences(run):
    """Start right after a sentence end, stop at a sentence end."""
    ends = [k for k, w in enumerate(run) if w["word"].endswith(SENT_END)]
    if not ends:
        return []
    i = run[0]["i"]
    clean_start = i == 0 or words[i - 1]["word"].endswith(SENT_END)
    return run[(0 if clean_start else ends[0] + 1):ends[-1] + 1]


runs = split_runs(words)
has_punct = sum(w["word"].endswith(SENT_END) for w in words) >= len(words) / 50
if WHOLE_SENTENCES and has_punct:
    runs = [r for r in map(whole_sentences, runs) if r]
elif WHOLE_SENTENCES:
    print("no sentence ends in the text -> cutting by pauses only")
print(len(runs), "runs")

no sentence ends in the text -> cutting by pauses only
14 runs


In [10]:
def cut_score(a, b):
    """How good a cut between word a and the next word b is. Higher = better, 7 at most."""
    pause = b["start"] - a["end"]
    score = 4 * min(pause, 1.0)
    if a["word"].endswith(SENT_END):
        score += 3
    elif a["word"].endswith(CLAUSE_END):
        score += 1
    if pause < 0.05:
        score -= 2
    return score


def best_cuts(run):
    """Dynamic programming: best[j] = cheapest way to cut the first j words."""
    n = len(run)
    best, prev = [0.0] + [float("inf")] * n, [0] * (n + 1)
    for j in range(1, n + 1):
        for i in range(j - 1, -1, -1):  # last clip = run[i:j]
            length = run[j - 1]["end"] - run[i]["start"] + 2 * PAD_SEC
            if length > MAX_SEC and i < j - 1:
                break
            cost = best[i] + 4 * ((length - TARGET_SEC) / TARGET_SEC) ** 2
            if length < MIN_SEC:
                cost += 5
            if j < n:  # we cut after word j-1 (the end of the run is free)
                cost += 7 - cut_score(run[j - 1], run[j])
            if cost < best[j]:
                best[j], prev[j] = cost, i
    groups, j = [], n
    while j > 0:
        groups.append(run[prev[j]:j])
        j = prev[j]
    return groups[::-1]

In [11]:
def clean_text(text):
    """One spelling for Uzbek apostrophes: oʻ gʻ with ʻ (U+02BB), tutuq belgisi with ʼ (U+02BC)."""
    out = []
    for k, ch in enumerate(text):
        if ch in "'‘’`ʼʻ":
            ch = "ʻ" if k > 0 and text[k - 1] in "oOgG" else "ʼ"
        out.append(ch)
    return " ".join("".join(out).split())


def make_clip(group):
    first, last, spk = group[0], group[-1], group[0]["speaker"]
    before = words[first["i"] - 1]["end"] if first["i"] > 0 else 0.0
    after = words[last["i"] + 1]["start"] if last["i"] + 1 < len(words) else total_sec
    start = max(first["start"] - PAD_SEC, (before + first["start"]) / 2, 0.0)
    end = min(last["end"] + PAD_SEC, (last["end"] + after) / 2, total_sec)
    for s, e, other in turns:
        if other == spk or overlap(start, end, s, e) == 0:
            continue
        if e <= first["start"]:
            start = max(start, e)  # other voice in the padding before
        elif s >= last["end"]:
            end = min(end, s)  # other voice in the padding after
        elif overlap(first["start"], last["end"], s, e) > 0.05:
            return None  # other voice inside the words
    text = clean_text(" ".join(w["word"] for w in group))
    return {"start": round(start, 3), "end": round(end, 3), "speaker": spk, "text": text}


clips, dropped = [], Counter()
for run in runs:
    for group in best_cuts(run):
        c = make_clip(group)
        if c is None:
            dropped["other voice inside"] += 1
        elif c["end"] - c["start"] < MIN_SEC:
            dropped["too short"] += 1
        else:
            clips.append(c)

kept = sum(c["end"] - c["start"] for c in clips)
print(len(clips), "clips,", round(kept, 1), "s kept, dropped:", dict(dropped))
for c in clips[:15]:
    print(f'{c["speaker"]}  {c["start"]:6.2f} - {c["end"]:6.2f}  ({c["end"] - c["start"]:4.1f}s)  {c["text"]}')

7 clips, 35.0 s kept, dropped: {'too short': 7}
speaker_0    0.08 -   2.74  ( 2.7s)  Oʻzbekistonda navin yutgan odam borligini bilarmidi
speaker_0    8.70 -  10.74  ( 2.0s)  borgan professionallik
speaker_0   15.50 -  29.38  (13.9s)  millatchi ular naviy bir paytida navin yutgan sekkan bitta xayol eʼtibor Nyukada turadi sayti boʻm oʻqiyotgan pastdan
speaker_0   35.32 -  40.66  ( 5.3s)  biror kartonda past-yu noutseker bilan gaplashganda aytganda navi Tepa
speaker_0   45.30 -  47.66  ( 2.4s)  Hamma besh tashlab ketardi-yu Nam yoqtirmayman
speaker_0   59.22 -  61.34  ( 2.1s)  oʻsha navini masalan bir oltiligin
speaker_0   62.58 -  69.19  ( 6.6s)  bor kibersport yurgani uchun oʻshalarni yutgan oʻzbek.


In [12]:
wav_dir = out_dir / "wavs"
wav_dir.mkdir(exist_ok=True)
for old in wav_dir.glob("*.wav"):
    old.unlink()

audio, sr = sf.read(audio_path, dtype="float32", always_2d=True)
audio = audio.mean(axis=1)  # cut from the original file, mono

with open(out_dir / "manifest.jsonl", "w", encoding="utf-8") as f:
    for n, c in enumerate(clips):
        name = f"{audio_path.stem}_{n:04d}_{c['speaker']}.wav"
        sf.write(wav_dir / name, audio[int(c["start"] * sr):int(c["end"] * sr)], sr)
        row = {"audio_filepath": f"wavs/{name}", "duration": round(c["end"] - c["start"], 3),
               "text": c["text"], "speaker": c["speaker"]}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("saved", len(clips), "clips to", wav_dir.relative_to(root))

saved 7 clips to outputs/chunks/OyQOvi9Qhxs/wavs
